In [1]:
%pip install ucimlrepo --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=601)
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]

rename_map = {
    "Type": "type",
    "Air temperature": "air_temp_k",
    "Process temperature": "process_temp_k",
    "Rotational speed": "rpm",
    "Torque": "torque_nm",
    "Tool wear": "tool_wear_min",
    "Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]
df.head()


,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [4]:
print("Missing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nTarget balance (share of rows):")
print(df["machine_failure"].value_counts(normalize=True).round(4))


Missing values per column:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
type                object
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance (share of rows):
0    0.9661
1    0.0339
Name: machine_failure, dtype: float64


In [5]:
df.to_csv("../data/ai4i_clean.csv", index=False)
print("Saved", df.shape, "to data/ai4i_clean.csv")


Saved (10000, 7) to data/ai4i_clean.csv


In [6]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "51a97c10-b319-4e79-9c52-fdbfb4652b5c"
RESOURCE_GROUP = "rg-mlab1-abdullah"
WORKSPACE = "mlw-lab1-abdullah"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: mlw-lab1-abdullah


In [7]:
import os
folder = os.path.abspath("../data/mltable_folder")
print(folder)
for root, dirs, files in os.walk(folder):
    print(root, files)

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-abdullah-lab1/code/Users/royalansari159/azureml-knn-maintenance/data/mltable_folder


In [8]:
%pip install mltable --quiet


Note: you may need to restart the kernel to use updated packages.


In [9]:
import os, shutil, mltable

folder = os.path.abspath("../data/mltable_v4")
os.makedirs(folder, exist_ok=True)
shutil.copy("../data/ai4i_clean.csv", os.path.join(folder, "ai4i_clean.csv"))

paths = [{"file": os.path.join(folder, "ai4i_clean.csv")}]
tbl = mltable.from_delimited_files(paths)
tbl.save(folder)

print(os.listdir(folder))
print(open(os.path.join(folder, "MLTable")).read())

['.amlignore', '.amlignore.amltmp', 'ai4i_clean.csv', 'MLTable']
paths:
- file: ai4i_clean.csv
transformations:
- read_delimited:
    delimiter: ','
    empty_as_string: false
    encoding: utf8
    header: all_files_same_headers
    include_path_column: false
    infer_column_types: true
    partition_size: 20971520
    path_column: Path
    support_multi_line: false
type: mltable



In [19]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

table_asset = Data(
    name="ai4i-maintenance-table",
    version="4",
    type=AssetTypes.MLTABLE,
    path=folder,
    description="AI4I clean data as MLTable for AutoML",
)
table_asset = ml_client.data.create_or_update(table_asset)
print("Registered:", table_asset.name, "version", table_asset.version)

HttpResponseError: (UserError) A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Code: UserError
Message: A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "b42024a43fa3649586ce1b5fbfdf08fb",
        "request": "076a132239654f85"
    }
}Type: Environment
Info: {
    "value": "eastus"
}Type: Location
Info: {
    "value": "eastus"
}Type: Time
Info: {
    "va

In [ ]:
import os, shutil, mltable

df_auto = df.copy()
df_auto["type"] = df_auto["type"].map({"L": 0, "M": 1, "H": 2}).astype(float)

folder5 = os.path.abspath("../data/mltable_v5")
os.makedirs(folder5, exist_ok=True)
df_auto.to_csv(os.path.join(folder5, "ai4i_numeric.csv"), index=False)

tbl = mltable.from_delimited_files([{"file": os.path.join(folder5, "ai4i_numeric.csv")}])
tbl.save(folder5)

print(os.listdir(folder5))
print(df_auto.dtypes)

In [17]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

table_asset = Data(
    name="ai4i-maintenance-table",
    version="5",
    type=AssetTypes.MLTABLE,
    path=folder5,
    description="AI4I data with numeric type column for AutoML KNN",
)
table_asset = ml_client.data.create_or_update(table_asset)
print("Registered:", table_asset.name, "version", table_asset.version)

NameError: name 'folder5' is not defined

In [ ]:
import mltable
tbl = mltable.load("azureml:ai4i-maintenance-table:5")
d = tbl.to_pandas_dataframe()
print(d.dtypes)
print(d.isna().sum())
print(d["type"].unique())

In [ ]:
import os, mltable
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

df7 = df.copy()
df7["type"] = df7["type"].map({"L": 0, "M": 1, "H": 2})
print("Missing in type:", df7["type"].isna().sum())   # must be 0

num_cols = ["type", "air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]
df7[num_cols] = df7[num_cols].astype("float64")
df7["machine_failure"] = df7["machine_failure"].astype("int64")
print(df7.dtypes)

folder7 = os.path.abspath("../data/mltable_v7")
os.makedirs(folder7, exist_ok=True)
df7.to_csv(os.path.join(folder7, "ai4i_v7.csv"), index=False)

tbl = mltable.from_delimited_files([{"file": os.path.join(folder7, "ai4i_v7.csv")}])
tbl = tbl.convert_column_types({
    "type": mltable.DataType.to_float(),
    "air_temp_k": mltable.DataType.to_float(),
    "process_temp_k": mltable.DataType.to_float(),
    "rpm": mltable.DataType.to_float(),
    "torque_nm": mltable.DataType.to_float(),
    "tool_wear_min": mltable.DataType.to_float(),
    "machine_failure": mltable.DataType.to_int(),
})
tbl.save(folder7)

asset = Data(name="ai4i-maintenance-table", version="7", type=AssetTypes.MLTABLE,
             path=folder7, description="All numeric, explicit column types")
asset = ml_client.data.create_or_update(asset)
print("Registered version", asset.version, os.listdir(folder7))

In [ ]:
d = mltable.load("azureml:ai4i-maintenance-table:7").to_pandas_dataframe()
print(d.dtypes)
print(d["type"].unique())

In [20]:
job = automl.classification(
    compute="cc-lab1",
    experiment_name="knn-predictive-maintenance-automl",
    training_data=Input(type="mltable", path="azureml:ai4i-maintenance-table:4"),
    target_column_name="machine_failure",
    primary_metric="AUC_weighted",
    n_cross_validations=5,
    enable_model_explainability=False,
)
job.set_limits(timeout_minutes=30, trial_timeout_minutes=15,
               max_trials=10, enable_early_termination=True)
job.set_training(allowed_training_algorithms=["KNN", "LogisticRegression"],
                 enable_vote_ensemble=False, enable_stack_ensemble=False)

submitted = ml_client.jobs.create_or_update(job)
print("Job name:", submitted.name)
print("Open in studio:", submitted.studio_url)

NameError: name 'automl' is not defined